In [1]:
import subprocess
subprocess.check_call(["pip", "install", "xgboost"])

0

IMPORT LIBRARIES

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report,roc_auc_score,confusion_matrix,ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import pickle

 LOAD CLEANED DATASET

In [3]:
df = pd.read_csv("startup_cleaned_dataset.csv")
print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (923, 42)


RECREATE TARGET VARIABLE

In [4]:
df["status"] = df["status"].map({"acquired": 1, "closed": 0})
print("Target variable distribution:")
print(df["status"].value_counts())

Target variable distribution:
status
1    597
0    326
Name: count, dtype: int64


ONE HOT ENCODING

In [5]:
df_encoded = pd.get_dummies(
    df, 
    columns=["state_code", "category_code"], 
    drop_first=True
)
print("Shape after encoding:", df_encoded.shape)

Shape after encoding: (923, 108)


In [6]:
X = df_encoded[[
    "funding_total_usd", "funding_rounds", "milestones",
    "relationships", "avg_participants", "is_top500",
    "age_first_funding_year", "age_last_funding_year",
    "age_first_milestone_year", "age_last_milestone_year",
    "has_VC", "has_angel", "has_roundA", "has_roundB",
    "has_roundC", "has_roundD",
    "is_software", "is_web", "is_mobile", "is_enterprise",
    "is_biotech", "is_CA", "is_NY", "is_MA", "is_TX"
]]
y = df_encoded["status"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (923, 25)
Target shape: (923,)


TRAIN TEST AND SPLIT

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training set size:", X_train.shape[0], "rows")
print("Testing set size:", X_test.shape[0], "rows")
print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTesting target distribution:")
print(y_test.value_counts())

Training set size: 738 rows
Testing set size: 185 rows

Training target distribution:
status
1    477
0    261
Name: count, dtype: int64

Testing target distribution:
status
1    120
0     65
Name: count, dtype: int64


FEATURE SCALING

In [8]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Scaling complete")


Scaling complete


TRAIN LOGISTIC REGRESSION

In [9]:
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_sc, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


EVALUATE LOGISTIC REGRESSION

In [10]:
y_pred_lr = lr_model.predict(X_test_sc)
y_prob_lr = lr_model.predict_proba(X_test_sc)[:, 1]
print("LOGISTIC REGRESSION RESULTS")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_prob_lr), 3))

LOGISTIC REGRESSION RESULTS
              precision    recall  f1-score   support

           0       0.57      0.78      0.66        65
           1       0.85      0.68      0.76       120

    accuracy                           0.72       185
   macro avg       0.71      0.73      0.71       185
weighted avg       0.76      0.72      0.73       185

ROC-AUC Score: 0.81


RANDOM FOREST 

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train_sc, y_train)
print("Number of trees built:", rf_model.n_estimators)

Number of trees built: 100


EVALUATE RANDOM FOREST

In [12]:
y_pred_rf = rf_model.predict(X_test_sc)
y_prob_rf = rf_model.predict_proba(X_test_sc)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_prob_rf), 3))

              precision    recall  f1-score   support

           0       0.73      0.57      0.64        65
           1       0.79      0.88      0.83       120

    accuracy                           0.77       185
   macro avg       0.76      0.73      0.74       185
weighted avg       0.77      0.77      0.77       185

ROC-AUC Score: 0.843


In [13]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

print("TOP 10 MOST IMPORTANT FEATURES:")
print(feature_importance.to_string(index=False))

TOP 10 MOST IMPORTANT FEATURES:
                 feature  importance
 age_last_milestone_year    0.154534
           relationships    0.135601
       funding_total_usd    0.105591
age_first_milestone_year    0.104445
   age_last_funding_year    0.087102
  age_first_funding_year    0.083103
        avg_participants    0.073910
              milestones    0.053427
          funding_rounds    0.038461
               is_top500    0.022763


XGBOOST TRAINING

In [14]:
xgb_model = XGBClassifier(
    scale_pos_weight=261/477,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train_sc, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


EVALUATE THE XGBOOST

In [17]:
y_pred_xgb = xgb_model.predict(X_test_sc)
y_prob_xgb = xgb_model.predict_proba(X_test_sc)[:, 1]
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_prob_xgb), 3))

              precision    recall  f1-score   support

           0       0.66      0.57      0.61        65
           1       0.78      0.84      0.81       120

    accuracy                           0.75       185
   macro avg       0.72      0.71      0.71       185
weighted avg       0.74      0.75      0.74       185

ROC-AUC Score: 0.814


USE PICKLE AND SAVE THE MODEL

In [18]:
# CELL 15 — Save the winning model
# We save Random Forest because it performed best
# pickle converts the trained model into a file we can load later in our app

with open("best_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("MODEL SELECTED: Random Forest")
print("Accuracy  : 77%")
print("ROC-AUC   : 0.843")
print("Weighted F1: 0.77")
print("Models saved successfully!")
print("best_model.pkl — trained Random Forest")
print("scaler.pkl     — fitted StandardScaler")

MODEL SELECTED: Random Forest
Accuracy  : 77%
ROC-AUC   : 0.843
Weighted F1: 0.77
Models saved successfully!
best_model.pkl — trained Random Forest
scaler.pkl     — fitted StandardScaler
